Este proyecto propone construir un modelo de clasificación supervisada que, a partir de datos demográficos y socioeconómicos de una persona adulta (edad, nivel educativo, ocupación, estado civil, país de origen, etc.), prediga si dicha persona ganará más o menos de 50,000 USD al año.

En base a los resultados del modelo, los estudiantes deberán desarrollar un sistema de recomendación interpretativo, capaz de sugerir posibles estrategias o cambios para aumentar la probabilidad de superar ese umbral de ingresos.

Objetivos

Explorar los datos del censo.

Construir perfiles socioeconómicos.

Explorar la importancia y peso de variables sociales (educación, género, raza, etc.) en predicciones económicas.

Aplicar técnicas de sistemas de recomendación.

Visualizar y comunicar hallazgos de forma profesional.

📝 Instrucciones

Carga del conjunto de datos. Usaremos el dataset Adult Income Dataset, también conocido como "Census Income" este información fue recolectada por la Oficina del Censo de EE.UU. y descargada por la academia para guardarla en esta carpeta de proyecto bajo el nombre adult-census-income.csv o puedes cargarlo en el código directamente desde el siguente enlace:# Explore here

1. EDA

In [71]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
import warnings


url="https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv"

df=pd.read_csv(url)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [72]:
df.info()
print("="*80)
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [73]:
df.shape

(32561, 15)

In [74]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [75]:
duplicados = df.duplicated()
num_duplicados = duplicados.sum()

print(f"El dataset tiene {num_duplicados} filas duplicadas")

df_duplicados = df[duplicados]
print("Filas duplicadas:")
print(df_duplicados)

# Eliminar duplicados
#usar el método drop.duplicated() para eliminar filas duplicadas

df = df.drop_duplicates().reset_index(drop=True)

print ("="*80)

print(f"Shape sin duplicados: {df.shape}")

El dataset tiene 24 filas duplicadas
Filas duplicadas:
       age         workclass  fnlwgt     education  education.num  \
8453    25           Private  308144     Bachelors             13   
8645    90           Private   52386  Some-college             10   
12202   21           Private  250051  Some-college             10   
14346   20           Private  107658  Some-college             10   
15603   25           Private  195994       1st-4th              2   
17344   21           Private  243368     Preschool              1   
19067   46           Private  173243       HS-grad              9   
20388   30           Private  144593       HS-grad              9   
20507   19           Private   97261       HS-grad              9   
22783   19           Private  138153  Some-college             10   
22934   19           Private  146679  Some-college             10   
23276   49           Private   31267       7th-8th              4   
23660   25           Private  195994       1st-4

Shape sin duplicados: (32537, 15)


In [76]:
df["workclass"].unique()

<StringArray>
[               '?',          'Private',        'State-gov',
      'Federal-gov', 'Self-emp-not-inc',     'Self-emp-inc',
        'Local-gov',      'Without-pay',     'Never-worked']
Length: 9, dtype: str

Se visualizan valores "?", se puede reemplazar con nulos

In [77]:
df = df.replace("?", np.nan)

In [78]:
df.isnull().sum()

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     582
income               0
dtype: int64

Se elimina las filas que tienen null ya que equivalen el 7% del dataset.

In [79]:
df = df.dropna()

In [80]:
df.shape

(30139, 15)

In [81]:
df.info()

<class 'pandas.DataFrame'>
Index: 30139 entries, 1 to 32536
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             30139 non-null  int64
 1   workclass       30139 non-null  str  
 2   fnlwgt          30139 non-null  int64
 3   education       30139 non-null  str  
 4   education.num   30139 non-null  int64
 5   marital.status  30139 non-null  str  
 6   occupation      30139 non-null  str  
 7   relationship    30139 non-null  str  
 8   race            30139 non-null  str  
 9   sex             30139 non-null  str  
 10  capital.gain    30139 non-null  int64
 11  capital.loss    30139 non-null  int64
 12  hours.per.week  30139 non-null  int64
 13  native.country  30139 non-null  str  
 14  income          30139 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [82]:
df["income"].value_counts()

income
<=50K    22633
>50K      7506
Name: count, dtype: int64

Pasar la variable objetivo a variable binaria

In [83]:
df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

Conclusiones:

La variable objetivo ssería income, se visualiza que hay más personas que tienen ingresos <=50K.
El dataset tiene 8 variables categóricas y 6 variables numéricas.

Procesamiento de datos

In [84]:
X=df.drop("income",axis=1)   #variable predictora
y=df["income"]               #variable objetivo


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
2433,55,State-gov,175127,Bachelors,13,Married-civ-spouse,Adm-clerical,Husband,White,Male,7688,0,38,United-States
4782,49,Local-gov,120451,10th,6,Separated,Other-service,Unmarried,Black,Female,0,0,40,United-States
5873,36,Private,188834,9th,5,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,50,United-States
29452,37,Private,99146,Bachelors,13,Married-civ-spouse,Sales,Husband,White,Male,0,0,60,United-States
30359,19,Private,174732,Some-college,10,Never-married,Other-service,Own-child,Black,Male,0,0,25,United-States


In [85]:
X_train.shape, X_test.shape

((24111, 14), (6028, 14))

Escalado

In [86]:
num_variables = [
    "age",
    "fnlwgt",
    "education.num",
    "capital.gain",
    "capital.loss",
    "hours.per.week"
]

In [87]:
# instancio el escalador
scaler = StandardScaler()

# entreno el escalador con los datos de entrenamiento
scaler.fit(X_train[num_variables])

# aplico el escalador en amhos
X_train_num_scal = scaler.transform(X_train[num_variables])
X_train_num_scal = pd.DataFrame(X_train_num_scal, index = X_train.index, columns = num_variables)

X_test_num_scal = scaler.transform(X_test[num_variables])
X_test_num_scal = pd.DataFrame(X_test_num_scal, index = X_test.index, columns = num_variables)

X_train_num_scal.head()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
2433,1.259355,-0.136473,1.124235,0.896711,-0.221265,-0.245948
4782,0.801909,-0.652764,-1.615607,-0.148596,-0.221265,-0.079549
5873,-0.189222,-0.007041,-2.007013,-0.148596,-0.221265,0.752448
29452,-0.112981,-0.853941,1.124235,-0.148596,-0.221265,1.584444
30359,-1.485317,-0.140203,-0.049983,-0.148596,-0.221265,-1.327544


In [88]:
# instancio el escalador
scaler = MinMaxScaler()

# entreno el escalador con los datos de entrenamiento

scaler.fit(X_train[num_variables])

# aplico el escalador en amhos
X_train_num_mm = scaler.transform(X_train[num_variables])
X_train_num_mm = pd.DataFrame(X_train_num_mm, index = X_train.index, columns = num_variables)

X_test_num_mm = scaler.transform(X_test[num_variables])
X_test_num_mm = pd.DataFrame(X_test_num_mm, index = X_test.index, columns = num_variables)

X_train_num_mm.head()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
2433,0.520548,0.109697,0.800000,0.076881,0.0,0.377551
4782,0.438356,0.072527,0.333333,0.000000,0.0,0.397959
5873,0.260274,0.119016,0.266667,0.000000,0.0,0.500000
29452,0.273973,0.058043,0.800000,0.000000,0.0,0.602041
30359,0.027397,0.109429,0.600000,0.000000,0.0,0.244898


Encoding

In [89]:
cat_variables = [
    "workclass",
    "education",
    "marital.status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native.country"
]

In [90]:
# instancio el encoder

onehot_encoder = OneHotEncoder(sparse_output=False)

# entreno el encoder con los datos de entrenamiento
onehot_encoder.fit(X_train[cat_variables])

# aplico el encoder en amhos
X_train_cat_ohe = onehot_encoder.transform(X_train[cat_variables])
X_train_cat_ohe = pd.DataFrame(X_train_cat_ohe, index = X_train.index, columns=onehot_encoder.get_feature_names_out(cat_variables))

X_test_cat_ohe = onehot_encoder.transform(X_test[cat_variables])
X_test_cat_ohe = pd.DataFrame(X_test_cat_ohe, index = X_test.index, columns=onehot_encoder.get_feature_names_out(cat_variables))

X_train_cat_ohe.head()

,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,education_10th,education_11th,education_12th,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
2433,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4782,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5873,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
29452,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
30359,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [91]:
# unificamos el dataset preprocesado hasta el momento
X_train_final = pd.concat([X_train_num_scal, X_train_cat_ohe], axis=1)
X_test_final = pd.concat([X_test_num_scal, X_test_cat_ohe], axis=1)

X_train_final.head()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,workclass_Federal-gov,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
2433,1.259355,-0.136473,1.124235,0.896711,-0.221265,-0.245948,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4782,0.801909,-0.652764,-1.615607,-0.148596,-0.221265,-0.079549,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5873,-0.189222,-0.007041,-2.007013,-0.148596,-0.221265,0.752448,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
29452,-0.112981,-0.853941,1.124235,-0.148596,-0.221265,1.584444,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
30359,-1.485317,-0.140203,-0.049983,-0.148596,-0.221265,-1.327544,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
